# Chapter 7: Support Vector Machines and Kernel Methods

## 📋 Summary

Support Vector Machines (SVMs) are powerful supervised learning models effective for both classification and regression in high-dimensional spaces. They work by finding the **optimal hyperplane** that maximally separates classes. The **kernel trick** extends SVMs to non-linear boundaries without explicitly computing high-dimensional transformations.

This chapter covers linear SVMs, kernel functions (RBF, polynomial, sigmoid), hyperparameter tuning (C and gamma), SVMs in high-dimensional spaces, and evaluation.

---

## 🧠 Theoretical Explanation

### Maximum Margin Classifier
An SVM finds the hyperplane `w·x + b = 0` that maximizes the **margin** — the distance between the hyperplane and the nearest training points (support vectors). Maximizing the margin minimizes generalization error.

**Hard-margin SVM** (linearly separable data):
`maximize: 2/||w||  subject to: yi(w·xi + b) ≥ 1`

**Soft-margin SVM** (allows misclassifications):
Introduces slack variables ξi and penalty C:
`minimize: ||w||²/2 + C·Σξi  subject to: yi(w·xi + b) ≥ 1 - ξi`

- Large C: small margin, fewer misclassifications (low bias, high variance)
- Small C: large margin, more misclassifications (high bias, low variance)

### The Kernel Trick
Instead of explicitly mapping data to higher dimensions, kernels compute dot products in the high-dimensional space implicitly:

- **Linear**: `K(x,z) = x·z`
- **RBF (Gaussian)**: `K(x,z) = exp(-γ||x-z||²)` — most popular; γ controls the width
- **Polynomial**: `K(x,z) = (x·z + r)^d`
- **Sigmoid**: `K(x,z) = tanh(γx·z + r)`

### SVR — Support Vector Regression
SVR finds a function that deviates from targets by at most ε (the epsilon-tube). Points outside the tube incur a penalty.


## 7.1 Linear SVM Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=42
)

# Linear SVM
svm_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='linear', C=1.0))
])
svm_linear.fit(X_train, y_train)
print('Linear SVM:')
print(classification_report(y_test, svm_linear.predict(X_test), target_names=cancer.target_names))

## 7.2 Kernel Functions

In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score

kernels = ['linear', 'rbf', 'poly', 'sigmoid']
results = []

for kernel in kernels:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel=kernel, C=1.0, random_state=42))
    ])
    score = cross_val_score(pipe, cancer.data, cancer.target, cv=5).mean()
    results.append({'Kernel': kernel, 'CV Accuracy': round(score, 4)})

pd.DataFrame(results)

## 7.3 Hyperparameter Tuning: C and gamma

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf'))])

param_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(cancer.data, cancer.target)

print('Best params:', grid.best_params_)
print('Best score:', round(grid.best_score_, 4))

## 7.4 Visualizing SVM Decision Boundary (2D)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, kernel in zip(axes, ['linear', 'rbf']):
    clf = Pipeline([('s', StandardScaler()), ('svm', SVC(kernel=kernel, C=1.0))])
    clf.fit(X_train, y_train)

    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 300),
                         np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 300))
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='RdYlBu', edgecolors='k', s=30)
    ax.set_title(f'SVM - {kernel} kernel (acc={clf.score(X_test, y_test):.2f})')

plt.tight_layout(); plt.show()

## 7.5 Support Vector Regression (SVR)

In [ ]:
from sklearn.svm import SVR
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

housing = fetch_california_housing()
X_train, X_test, y_train, y_test = train_test_split(
    housing.data[:2000], housing.target[:2000], test_size=0.2, random_state=42
)

svr = Pipeline([('s', StandardScaler()), ('svr', SVR(kernel='rbf', C=1.0, epsilon=0.1))])
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)

print(f'SVR RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')
print(f'SVR R²:   {r2_score(y_test, y_pred):.4f}')

## 🔑 Key Takeaways

- SVMs find the **maximum margin hyperplane** — the decision boundary that is furthest from the nearest training points.
- **C** controls the tradeoff between margin size and misclassification: large C = narrow margin, low C = wide margin.
- The **kernel trick** maps data to higher dimensions implicitly, enabling non-linear decision boundaries.
- **RBF kernel** is the go-to choice for most non-linear problems; tune C and γ together.
- **Always scale features** — SVMs are very sensitive to feature scale.
- SVMs work well in high-dimensional spaces (text, genomics) even with few samples.
